
# Disneyland QA Demo

This notebook is the live demo surface for the adaptive analytical RAG system. It is designed for interviewer review, manual experimentation, and lightweight evaluation.



## Demo Flow

1. Edit `QUESTION`
2. Run the QA engine once
3. Inspect the structured plan
4. Inspect deterministic analytics
5. Inspect metadata-filtered Chroma evidence
6. Inspect the final grounded answer input preview
7. Read the answer and optional judge result


In [1]:

import json
import os
import warnings
from pathlib import Path
from textwrap import dedent, shorten

import pandas as pd
from IPython.display import JSON, Markdown, display

from src.config import get_settings
from src.data import load_clean_reviews
from src.evaluation import (
    build_judge_evaluator,
    evaluate_case_deterministically,
    evaluate_single_result,
    load_evaluation_cases,
)
from src.prompts import build_answer_payload
from src.qa import QAEngineResult, build_qa_engine
from src.vectorstore import build_or_load_vectorstore

warnings.filterwarnings('ignore', category=UserWarning)
pd.options.display.max_columns = 60
pd.options.display.max_colwidth = 160
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')


/Users/mhdaldahmani/venvs/ai-interview/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'false'

In [ ]:

settings = get_settings()
DATASET_PATH = Path('DisneylandReviews.csv')
reviews_df = load_clean_reviews(DATASET_PATH)
vectorstore_result = build_or_load_vectorstore(reviews_df, settings=settings, verbose=False)
engine = build_qa_engine(
    review_df=reviews_df,
    vectorstore=vectorstore_result.vectorstore,
    settings=settings,
    verbose_index=False,
)
manifest = vectorstore_result.manifest

dated_reviews = reviews_df['Review_Date'].dropna()
setup_summary = pd.DataFrame(
    [
        ('Dataset review count', f"{len(reviews_df):,}"),
        ('Dataset date range', f"{dated_reviews.min():%Y-%m} to {dated_reviews.max():%Y-%m}"),
        ('Embedding model', manifest.get('local_embedding_model') or manifest.get('openai_embedding_model', 'Unavailable')),
        ('Planner model', settings.planner_model),
        ('Answer model', settings.answer_model),
        ('Judge model', settings.judge_model),
        ('Chroma index status', 'Reused existing index' if vectorstore_result.reused_existing_index else 'Built new index'),
    ],
    columns=['Item', 'Value'],
)

display(setup_summary)
if not settings.openai_api_key:
    display(Markdown('**OpenAI API key not detected.** Interactive answers and judge runs will return a clear error until `OPENAI_API_KEY` is configured.'))


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8754.78it/s]



## How The System Works

1. The planner converts the question into a structured plan.
2. The plan is validated and normalized.
3. `pandas` performs the required calculations.
4. Analytics determines which subsets need qualitative evidence.
5. Chroma retrieves semantically relevant reviews with metadata filters.
6. The answer model receives only calculated metrics and validated excerpts.
7. A separate judge model can evaluate the final answer.


In [ ]:

def _model_json(model):
    if model is None:
        return None
    if hasattr(model, 'model_dump'):
        return model.model_dump(mode='json')
    return model


def _compact_json(model):
    payload = _model_json(model)
    return payload if payload is not None else {}


def _analytics_table(result):
    analytics = result.analytics_result
    if analytics is None:
        return pd.DataFrame()
    if analytics.comparison_rows:
        rows = pd.DataFrame([row.model_dump(mode='json') for row in analytics.comparison_rows])
        if 'month_name' in rows.columns and rows['month_name'].notna().any():
            rows['Period'] = rows['month_name'].fillna(rows.get('group_key'))
        elif 'season' in rows.columns and rows['season'].notna().any():
            rows['Period'] = rows['season'].fillna(rows.get('group_key'))
        elif 'branch' in rows.columns:
            rows['Period'] = rows['branch']
        else:
            rows['Period'] = rows.get('group_key', '')

        if 'visit_score' in rows.columns and rows['visit_score'].notna().any():
            rank_source = rows.loc[rows['eligible'].fillna(False), ['Period', 'visit_score']].sort_values('visit_score', ascending=False)
            rank_map = {period: rank + 1 for rank, period in enumerate(rank_source['Period'])}
            rows['Rank'] = rows['Period'].map(rank_map)
        elif 'ranking_metric_value' in rows.columns and rows['ranking_metric_value'].notna().any():
            ascending = False
            if result.plan is not None:
                if result.plan.ranking_direction == 'lowest':
                    ascending = True
                elif result.plan.ranking_direction == 'best' and analytics.ranking_criterion in {'crowding_complaint_rate', 'negative_share'}:
                    ascending = True
            rank_source = rows.sort_values('ranking_metric_value', ascending=ascending)
            rank_map = {period: rank + 1 for rank, period in enumerate(rank_source['Period'])}
            rows['Rank'] = rows['Period'].map(rank_map)

        keep = [
            col for col in [
                'Period', 'review_count', 'average_rating', 'positive_share', 'negative_share',
                'crowding_complaint_rate', 'aspect_mention_rate', 'eligible', 'visit_score',
                'ranking_metric_value', 'Rank'
            ] if col in rows.columns
        ]
        return rows[keep].round(4)

    metrics = analytics.metrics.copy()
    baseline = analytics.baseline_metrics.copy()
    metric_rows = []
    if metrics:
        metric_rows.append({'Scope': 'Subset', **metrics})
    if baseline:
        metric_rows.append({'Scope': 'Baseline', **baseline})
    return pd.DataFrame(metric_rows).round(4)


def _retrieval_tasks_frame(tasks):
    if not tasks:
        return pd.DataFrame(columns=['task_id', 'purpose', 'query', 'branch', 'reviewer_location', 'month', 'season', 'rating_filter', 'top_k'])
    return pd.DataFrame([
        {
            'task_id': task.task_id,
            'purpose': task.purpose,
            'query': task.query,
            'branch': task.branch,
            'reviewer_location': task.reviewer_location,
            'month': task.month,
            'season': task.season,
            'rating_filter': task.rating_filter,
            'top_k': task.top_k,
        }
        for task in tasks
    ])


def _evidence_frame(evidence):
    rows = []
    for item in evidence:
        rows.append(
            {
                'Review ID': item.review_id,
                'Purpose': ', '.join(item.matched_purposes),
                'Rating': item.rating,
                'Branch': item.branch,
                'Reviewer location': item.reviewer_location,
                'Month': item.month,
                'Season': item.season,
                'Relevance': round(item.relevance_score, 4) if item.relevance_score is not None else None,
                'Validation': 'Valid' if item.validation_passed else 'Invalid',
                'Excerpt': shorten(item.text, width=140, placeholder='...'),
            }
        )
    return pd.DataFrame(rows)


def _answer_preview_markdown(result):
    payload = result.answer_payload
    if payload is None and result.plan is not None and result.analytics_result is not None:
        payload = build_answer_payload(
            question=result.question,
            plan=result.plan,
            analytics_result=result.analytics_result,
            evidence=result.evidence,
        )
    if payload is None:
        return '_No final answer payload is available._'
    return dedent(f'''
    **Question**

    {payload['question']}

    **Normalized plan**

    ```json
    {payload['plan_json']}
    ```

    **Authoritative statistics**

    ```text
    {payload['deterministic_summary']}
    ```

    **Retrieved review excerpts**

    ```text
    {payload['evidence_block']}
    ```

    **Unsupported-information warning**

    `{payload['unsupported_context']}`

    **Dataset date range**

    `{payload['date_range']}`
    ''')


def display_result_trace(question, result):
    display(Markdown('## A. Original Question'))
    display(Markdown(f'`{question}`'))

    if result.error:
        display(Markdown('## J. Errors'))
        display(Markdown(f'**{result.error}**'))
        return

    display(Markdown('## B. Raw Planner Output'))
    display(JSON(_compact_json(result.raw_plan), expanded=False))

    display(Markdown('## C. Normalized Plan'))
    display(JSON(_compact_json(result.plan), expanded=False))

    analytics = result.analytics_result
    route_summary = pd.DataFrame(
        [
            {
                'Selected analytics function': analytics.selected_analytics_function if analytics else None,
                'Applied filters': json.dumps(analytics.applied_filters, ensure_ascii=False) if analytics else None,
                'Sample size': analytics.sample_size if analytics else None,
                'Baseline metrics available': bool(analytics and analytics.baseline_metrics),
                'Warnings': ' | '.join(analytics.warnings) if analytics and analytics.warnings else 'None',
            }
        ]
    )
    display(Markdown('## D. Analytics Route'))
    display(route_summary)

    display(Markdown('## E. Deterministic Analytical Output'))
    display(_analytics_table(result))

    display(Markdown('## F. Resolved Retrieval Tasks'))
    display(_retrieval_tasks_frame(result.retrieval_tasks))

    display(Markdown('## G. Retrieved Evidence'))
    validation = result.retrieval_validation
    validation_summary = pd.DataFrame(
        [
            {
                'Validation status': 'Valid' if validation is None or validation.invalid_count == 0 else 'Needs review',
                'Unique reviews': len({item.review_id for item in result.evidence}),
                'Valid evidence': validation.valid_count if validation else len(result.evidence),
                'Invalid evidence': validation.invalid_count if validation else 0,
                'Warnings': ' | '.join((result.retrieval_warnings or []) + (validation.warnings if validation else [])) or 'None',
            }
        ]
    )
    display(validation_summary)
    display(_evidence_frame(result.evidence))

    display(Markdown('## H. Final Model Input Preview'))
    display(Markdown(_answer_preview_markdown(result)))

    display(Markdown('## I. Final Answer'))
    structured = result.structured_answer
    if structured is not None:
        display(Markdown('**Direct answer**'))
        display(Markdown(structured.direct_answer))
        display(pd.DataFrame({'Supporting metrics': structured.supporting_metrics}))
        display(pd.DataFrame({'Qualitative evidence': structured.evidence}))
        display(pd.DataFrame({'Limitations': structured.limitations}))
    display(Markdown('**Rendered answer**'))
    display(Markdown(result.answer or '_No rendered answer available._'))


## Interactive Question

In [ ]:
QUESTION = 'What is the best time of year to visit Hong Kong Disneyland?'

In [ ]:

if settings.openai_api_key:
    result = engine.ask(QUESTION, return_trace=True)
else:
    result = QAEngineResult(
        question=QUESTION,
        answer=None,
        raw_plan=None,
        plan=None,
        analytics_result=None,
        retrieval_tasks=[],
        evidence=[],
        error='OPENAI_API_KEY is not configured.',
        structured_answer=None,
        retrieval_validation=None,
        retrieval_warnings=[],
        answer_payload=None,
        trace=None,
    )


In [ ]:
display_result_trace(QUESTION, result)

## Ten-Question Showcase

In [ ]:

SHOWCASE_QUESTIONS = [
    'What do visitors from Australia say about Hong Kong Disneyland?',
    'What do visitors from the United Kingdom say about Disneyland Paris?',
    'Is the staff in Paris friendly?',
    'What do visitors dislike about food in California?',
    'Is Hong Kong Disneyland crowded?',
    'What is the best time of year to visit Hong Kong Disneyland?',
    'Which month has the fewest crowding complaints in California?',
    'How does Paris perform in summer compared with winter?',
    'Which park has the strongest customer satisfaction?',
    'Considering Hong Kong weather and public holidays, what is the best month to visit?',
]


In [ ]:

showcase_results = []
for question in SHOWCASE_QUESTIONS:
    if settings.openai_api_key:
        qa_result = engine.ask(question)
    else:
        qa_result = QAEngineResult(
            question=question,
            answer=None,
            raw_plan=None,
            plan=None,
            analytics_result=None,
            retrieval_tasks=[],
            evidence=[],
            error='OPENAI_API_KEY is not configured.',
            structured_answer=None,
            retrieval_validation=None,
            retrieval_warnings=[],
            answer_payload=None,
            trace=None,
        )
    showcase_results.append(qa_result)

showcase_summary = pd.DataFrame(
    [
        {
            'Question': question,
            'Intent': qa_result.plan.intent if qa_result.plan else None,
            'Branch': qa_result.plan.branch if qa_result.plan else None,
            'Analytics route': qa_result.analytics_result.selected_analytics_function if qa_result.analytics_result else None,
            'Sample size': qa_result.analytics_result.sample_size if qa_result.analytics_result else None,
            'Evidence count': len(qa_result.evidence),
            'Answer preview': shorten((qa_result.answer or qa_result.error or ''), width=120, placeholder='...'),
            'Error': qa_result.error,
        }
        for question, qa_result in zip(SHOWCASE_QUESTIONS, showcase_results)
    ]
)

display(showcase_summary)


In [ ]:
CASE_INDEX = 0

In [ ]:

selected_question = SHOWCASE_QUESTIONS[CASE_INDEX]
selected_result = engine.ask(selected_question, return_trace=True) if settings.openai_api_key else showcase_results[CASE_INDEX]
display_result_trace(selected_question, selected_result)


## Deterministic Evaluation

In [ ]:

evaluation_cases = load_evaluation_cases()
case_lookup = {case.question: case for case in evaluation_cases}

deterministic_rows = []
for question, qa_result in zip(SHOWCASE_QUESTIONS, showcase_results):
    case = case_lookup.get(question)
    if case is None:
        continue
    evaluation = evaluate_case_deterministically(case, qa_result)
    deterministic_rows.append(
        {
            'Case ID': case.id,
            'Intent correct': evaluation['intent_correct'],
            'Filters correct': evaluation['filters_correct'],
            'Comparison dimension correct': evaluation.get('comparison_dimension_correct'),
            'Analytics route correct': evaluation['analytics_route_correct'],
            'Retrieval filters valid': evaluation['retrieval_filters_valid'],
            'External-context behavior correct': evaluation['external_context_behavior_correct'],
            'Deterministic pass rate': evaluation['deterministic_pass_rate'],
            'Failure reason': evaluation['failure_reason'],
        }
    )

deterministic_eval_df = pd.DataFrame(deterministic_rows)
display(deterministic_eval_df)

summary_line = (
    f"Cases: {len(deterministic_eval_df)} | "
    f"Mean deterministic pass rate: {deterministic_eval_df['Deterministic pass rate'].mean():.3f} | "
    f"Failed case IDs: {', '.join(deterministic_eval_df.loc[deterministic_eval_df['Deterministic pass rate'] < 1.0, 'Case ID'].tolist()) or 'None'}"
)
display(Markdown(summary_line))



## Optional LLM-As-A-Judge

> The judge evaluates whether the answer is supported by the supplied dataset evidence. It is not treated as an unquestionable ground truth and should be combined with deterministic checks and human review.


In [ ]:

RUN_SINGLE_JUDGE = False
judge_result = None

if RUN_SINGLE_JUDGE and settings.openai_api_key and result.error is None:
    judge_result = evaluate_single_result(
        question=QUESTION,
        result=result,
        expected_behavior=case_lookup.get(QUESTION),
        settings=settings,
    )
    judge_summary = pd.DataFrame(
        [
            {
                'Relevance': judge_result.relevance,
                'Faithfulness': judge_result.faithfulness,
                'Completeness': judge_result.completeness,
                'Clarity': judge_result.clarity,
                'Limitation handling': judge_result.limitation_handling,
                'External-knowledge leak': judge_result.external_knowledge_leak,
            }
        ]
    )
    display(judge_summary)
    display(pd.DataFrame({'Unsupported claims': judge_result.unsupported_claims or ['None']}))
    display(Markdown(judge_result.explanation))
else:
    display(Markdown('Set `RUN_SINGLE_JUDGE = True` to judge the current interactive question.'))


In [ ]:

RUN_LLM_JUDGE = False
MAX_JUDGE_CASES = 5

judge_rows = []
if RUN_LLM_JUDGE and settings.openai_api_key:
    judge_evaluator = build_judge_evaluator(settings)
    judged = 0
    for question, qa_result in zip(SHOWCASE_QUESTIONS, showcase_results):
        if qa_result.error is not None or judged >= MAX_JUDGE_CASES:
            continue
        case = case_lookup.get(question)
        judge_result = evaluate_single_result(
            question=question,
            result=qa_result,
            expected_behavior=case,
            judge_evaluator=judge_evaluator,
            settings=settings,
        )
        judge_rows.append(
            {
                'Question': question,
                'Judge relevance': judge_result.relevance,
                'Judge faithfulness': judge_result.faithfulness,
                'Judge completeness': judge_result.completeness,
                'Judge clarity': judge_result.clarity,
                'Limitation handling': judge_result.limitation_handling,
                'External-knowledge leak': judge_result.external_knowledge_leak,
                'Final judge score': round((
                    judge_result.relevance
                    + judge_result.faithfulness
                    + judge_result.completeness
                    + judge_result.clarity
                    + judge_result.limitation_handling
                ) / 5, 3),
            }
        )
        judged += 1
    judge_eval_df = pd.DataFrame(judge_rows)
    display(judge_eval_df)
    if not judge_eval_df.empty:
        display(Markdown(
            f"Mean judge score: {judge_eval_df['Final judge score'].mean():.3f} | External-knowledge leaks: {int(judge_eval_df['External-knowledge leak'].fillna(False).astype(bool).sum())}"
        ))
else:
    display(Markdown('LLM judge is disabled by default. Set `RUN_LLM_JUDGE = True` to score up to `MAX_JUDGE_CASES` benchmark questions.'))
